# Experimental Model Training Methodology

Documents the exact executed matrix and hyperparameters. Full retraining is intentionally opt-in and invokes the authoritative pipeline rather than toy replacement logic.

**Safety:** default execution is lightweight. Expensive full-data work runs only behind an explicit environment flag.

In [ ]:
from pathlib import Path
import os, json
import pandas as pd

candidate = Path(os.environ.get('ASSISTMENTS_WORKSPACE', Path.cwd().parents[1] / 'ASSISTments Dataset')).resolve()
if not candidate.exists():
    raise FileNotFoundError('Set ASSISTMENTS_WORKSPACE to the local ASSISTments Dataset folder')
WORKSPACE = candidate
print('Using workspace:', WORKSPACE)

In [ ]:
step3 = WORKSPACE / '03_Experimental_Modeling'
registry = pd.read_csv(step3 / 'outputs' / 'ASSISTments_Step3_Experiment_Registry.csv')
display(registry[['experiment_id','feature_count','transformed_feature_count','regularization/hyperparameters','train_rows','validation_rows','test_rows','runtime_seconds','status']])
assert len(registry) == 8 and set(registry.status) == {'COMPLETE'}
print('Executed matrix verified: 6 LR + 2 H2 GB experiments')

In [ ]:
print('Shared preprocessing: train-frequency template encoding; train-fitted one-hot categories; numeric median imputation and standardization.')
print('RT ON: log1p and robust train-only contextualization. RT OFF: removes only four tagged historical RT fields.')
print('Split seed: 20260928; learners are disjoint across 70/15/15 partitions.')

In [ ]:
import subprocess, sys
RUN_FULL_EXPERIMENTS = os.environ.get('RUN_FULL_EXPERIMENTS') == '1'
if RUN_FULL_EXPERIMENTS:
    print('EXPENSIVE RUN AUTHORIZED: this rebuilds partitions, preprocessing, eight models, predictions and reports.')
    subprocess.run([sys.executable, str(step3 / 'run_step3_pipeline.py')], cwd=step3, check=True)
else:
    print('Full training skipped. To reproduce deliberately, set RUN_FULL_EXPERIMENTS=1 after confirming RAM, disk, dependencies and runtime budget.')